# Deteksi Kematangan Tomat (tomatOD) — 4 Skenario YOLO26

**Mata Kuliah**: Sains Data Terapan · **Kelompok 2** · **Arsitektur**: YOLO26 (Object Detection, `yolo26n`)
**Dataset**: tomatOD (Dataset Ninja), CC BY-NC-SA 4.0, 277 gambar, 3 kelas: `unripe`, `semi-ripe`, `fully-ripe`. Sudah dikonversi ke format YOLO dan ada di folder `data/tomatod` pada repo ini.

| # | Skenario | Cara |
|:--|:--|:--|
| 1 | Inference pretrained | `yolo26n.pt` (COCO) tanpa training. Karena COCO tidak punya kelas tomat/kematangan, yang diukur adalah **kemampuan melokalisasi** (class-agnostic) |
| 2 | Fine-tune: ubah hyperparameter, **head only** | load pretrained, `freeze` semua layer kecuali head, variasi optimizer dan `lr0` |
| 3 | Transfer learning: freeze sebagian layer | load pretrained, `freeze=N` untuk beberapa nilai N |
| 4 | Train from scratch | `yolo26n.yaml`, bobot acak |

Semua skenario memakai data, `imgsz`, `epochs`, `batch`, dan `seed` yang sama. Metrik utama: **mAP50** dan **mAP50-95** di test set (55 gambar, 844 objek).

> Runtime -> Change runtime type -> **T4 GPU** sebelum menjalankan.

## 0. Setup

In [ ]:
import torch
print('GPU tersedia:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('GPU tidak aktif! Training akan sangat lambat.')

In [ ]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

In [ ]:
# ===== KONFIGURASI (dipakai SEMUA skenario) =====
REPO_URL = 'https://github.com/januarsyah901/tomato-maturity-classification-yolov26.git'
SEED     = 42
IMGSZ    = 960     # tomat kecil di gambar 2000x2000, jadi pakai resolusi lebih tinggi dari default 640
EPOCHS   = 50
BATCH    = 16
PATIENCE = 15
PROJECT  = '/content/runs/tomatod'
PRETRAINED = 'yolo26n.pt'
SCRATCH    = 'yolo26n.yaml'

## 1. Ambil dataset dari GitHub

In [ ]:
import os, shutil, yaml
from pathlib import Path

REPO = Path('/content/repo')
shutil.rmtree(REPO, ignore_errors=True)
!git clone -q --depth 1 {REPO_URL} {REPO}

DATA_DIR = REPO/'data'/'tomatod'
cfg = yaml.safe_load(open(DATA_DIR/'data.yaml'))
cfg['path'] = str(DATA_DIR)           # path absolut
DATA = '/content/tomatod.yaml'
yaml.safe_dump(cfg, open(DATA, 'w'))

CLASSES = [cfg['names'][i] for i in sorted(cfg['names'])]
print('Kelas:', CLASSES)
for sp in ['train', 'val', 'test']:
    n_img = len(list((DATA_DIR/'images'/sp).glob('*.jpg')))
    cnt = [0]*len(CLASSES)
    for f in (DATA_DIR/'labels'/sp).glob('*.txt'):
        for line in f.read_text().split('\n'):
            if line.strip(): cnt[int(line.split()[0])] += 1
    print(f'{sp:<6} {n_img:>4} gambar   objek per kelas: ' + ', '.join(f'{c}={n}' for c, n in zip(CLASSES, cnt)))

> **Catatan kebocoran data**: semua gambar berasal dari beberapa sesi foto yang sama, jadi frame di train dan test bisa mirip. Skor test cenderung lebih optimis daripada kondisi di greenhouse atau sesi lain. Sebutkan ini di laporan.

## 2. Helper evaluasi

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

RESULTS = []

def record(tag, scenario, config, **m):
    row = dict(tag=tag, scenario=scenario, config=config, **m)
    RESULTS.append(row)
    print(f"[{tag}] " + '  '.join(f'{k}={v:.4f}' for k, v in m.items() if isinstance(v, float)) + f"  ({config})")
    return row

def evaluate_weights(weights, tag, scenario, config):
    m = YOLO(str(weights))
    r = m.val(data=DATA, split='test', imgsz=IMGSZ, batch=BATCH, verbose=False, plots=True,
              project=PROJECT, name=f'{tag}_test', exist_ok=True)
    per_class = {CLASSES[int(i)]: float(a) for i, a in zip(r.box.ap_class_index, r.box.ap50)}
    return record(tag, scenario, config,
                  mAP50=float(r.box.map50), mAP50_95=float(r.box.map),
                  precision=float(r.box.mp), recall=float(r.box.mr),
                  **{f'AP50_{c}': per_class.get(c, float('nan')) for c in CLASSES})

def train_run(tag, scenario, weights, config_desc, **kw):
    model = YOLO(weights)
    model.train(data=DATA, imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH, patience=PATIENCE,
                seed=SEED, project=PROJECT, name=tag, exist_ok=True, plots=True,
                verbose=False, **kw)
    best = Path(model.trainer.save_dir)/'weights'/'best.pt'
    return evaluate_weights(best, tag, scenario, config_desc)

---
## Skenario 1 — Inference pretrained (tanpa training)

`yolo26n.pt` dilatih di COCO (80 kelas), tidak ada kelas tomat apalagi tingkat kematangan. Jadi mAP per kelas tidak bermakna (hampir 0).
Yang bisa diukur secara adil adalah **lokalisasi class-agnostic**: apakah kotak yang diprediksi (kelas COCO apa pun) menimpa kotak tomat sebenarnya (IoU >= 0.5)? Kita hitung recall dan precision lokalisasi, dengan `conf=0.05` dan `max_det=300`.

In [ ]:
from ultralytics.utils.metrics import box_iou
pre = YOLO(PRETRAINED)
test_imgs = sorted((DATA_DIR/'images'/'test').glob('*.jpg'))

def load_gt(img):
    from PIL import Image
    W, H = Image.open(img).size
    rows = [l.split() for l in (DATA_DIR/'labels'/'test'/f'{img.stem}.txt').read_text().split('\n') if l.strip()]
    b = torch.tensor([[float(x) for x in r[1:]] for r in rows]) if rows else torch.zeros((0, 4))
    xyxy = torch.stack([(b[:,0]-b[:,2]/2)*W, (b[:,1]-b[:,3]/2)*H, (b[:,0]+b[:,2]/2)*W, (b[:,1]+b[:,3]/2)*H], 1) if len(b) else b
    return xyxy

tp_gt = n_gt = tp_pred = n_pred = 0
for img in test_imgs:
    gt = load_gt(img)
    r = pre.predict(str(img), conf=0.05, max_det=300, imgsz=IMGSZ, verbose=False)[0]
    pr = r.boxes.xyxy.cpu()
    n_gt += len(gt); n_pred += len(pr)
    if len(gt) and len(pr):
        iou = box_iou(gt, pr)
        tp_gt   += int((iou.max(1).values >= 0.5).sum())   # GT yang tertutup prediksi
        tp_pred += int((iou.max(0).values >= 0.5).sum())   # prediksi yang menimpa GT
recall_loc, prec_loc = tp_gt/max(n_gt,1), tp_pred/max(n_pred,1)
record('s1_pretrained_lokalisasi', '1. Pretrained inference', 'COCO yolo26n.pt, class-agnostic, IoU>=0.5',
       mAP50=0.0, mAP50_95=0.0, precision=prec_loc, recall=recall_loc,
       **{f'AP50_{c}': 0.0 for c in CLASSES})
print(f'\nLokalisasi: {tp_gt}/{n_gt} tomat tertutup prediksi (recall={recall_loc*100:.1f}%), precision={prec_loc*100:.1f}%')
print('mAP per kelas dicatat 0.0 karena COCO tidak punya kelas kematangan tomat.')

In [ ]:
# contoh visual prediksi pretrained
import random
random.seed(SEED)
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, img in zip(axes, random.sample(test_imgs, 3)):
    ax.imshow(pre.predict(str(img), conf=0.25, imgsz=IMGSZ, verbose=False)[0].plot()[..., ::-1]); ax.axis('off')
    ax.set_title(img.name, fontsize=8)
plt.tight_layout(); plt.show()

---
## Skenario 2 — Fine-tune: ubah hyperparameter, **head only**

Bobot COCO dimuat; seluruh layer di depan head di-freeze, hanya head deteksi yang dilatih (head dibuat ulang untuk 3 kelas). Hyperparameter divariasikan (optimizer dan `lr0`).

In [ ]:
n_layers = len(YOLO(PRETRAINED).model.model)
print('Jumlah modul:', n_layers, '(head = index %d)' % (n_layers-1))
HEAD_FREEZE = n_layers - 1

HP_GRID = [
    dict(optimizer='SGD',   lr0=0.01),
    dict(optimizer='SGD',   lr0=0.001),
    dict(optimizer='AdamW', lr0=0.001),
    dict(optimizer='AdamW', lr0=0.0001),
]
for hp in HP_GRID:
    train_run(f"s2_head_{hp['optimizer']}_lr{hp['lr0']}", '2. Fine-tune head only', PRETRAINED,
              f"freeze={HEAD_FREEZE} (head only), {hp['optimizer']}, lr0={hp['lr0']}",
              freeze=HEAD_FREEZE, pretrained=True, **hp)

---
## Skenario 3 — Transfer learning: load pretrained, freeze sebagian layer

`freeze=N` membekukan **N layer pertama**, sisanya ikut dilatih. Variasi: 0 (full fine-tune), 10 (kira-kira backbone), dan 3/4 layer. Optimizer dan `lr0` sama di semua run (SGD, 0.01).

> Poin 3 di soal terpotong logo. Kalau ada instruksi tambahan, tambahkan di cell ini.

In [ ]:
FREEZE_VARIANTS = {'none (full fine-tune)': 0, 'backbone saja (10)': 10, 'freeze 3/4': (n_layers*3)//4}
for name, n in FREEZE_VARIANTS.items():
    train_run(f's3_freeze{n}', '3. Transfer learning (freeze sebagian)', PRETRAINED,
              f'freeze={n} ({name}), SGD, lr0=0.01',
              freeze=n, pretrained=True, optimizer='SGD', lr0=0.01)

---
## Skenario 4 — Train from scratch

Arsitektur `yolo26n` dengan **bobot acak** (tanpa COCO). Hyperparameter sama dengan skenario 3. Dengan hanya 189 gambar train, hasilnya diperkirakan jauh di bawah skenario berbasis pretrained.

In [ ]:
train_run('s4_scratch', '4. Train from scratch', SCRATCH, 'bobot acak, SGD, lr0=0.01',
          pretrained=False, optimizer='SGD', lr0=0.01)

---
## Perbandingan semua skenario

In [ ]:
df = pd.DataFrame(RESULTS)
show = df[['scenario', 'tag', 'config', 'mAP50', 'mAP50_95', 'precision', 'recall'] + [f'AP50_{c}' for c in CLASSES]].copy()
for c in show.columns[3:]:
    show[c] = (show[c]*100).round(2)
pd.set_option('display.max_colwidth', 70); pd.set_option('display.width', 250)
display(show)
show.to_csv('/content/hasil_perbandingan.csv', index=False)

plt.figure(figsize=(10, 0.5*len(df)+2))
plt.barh(df.tag[::-1], df.mAP50[::-1]*100)
plt.xlabel('mAP50 test (%)'); plt.xlim(0, 100); plt.grid(axis='x', alpha=.3)
for y, v in enumerate(df.mAP50[::-1]*100):
    plt.text(v+0.5, y, f'{v:.1f}', va='center', fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
best_per = df.loc[df.groupby('scenario')['mAP50'].idxmax()][['scenario', 'tag', 'config', 'mAP50', 'mAP50_95']]
display(best_per)

In [ ]:
# confusion matrix dan kurva training model terbaik per skenario (hasil training saja)
from IPython.display import Image, display as _d
for tag in best_per.tag:
    if tag.startswith('s1'): continue
    print('='*70); print(tag)
    for fn in ['confusion_matrix_normalized.png', 'BoxPR_curve.png']:
        p = Path(PROJECT)/f'{tag}_test'/fn
        if p.exists(): _d(Image(str(p), width=600))
    p = Path(PROJECT)/tag/'results.png'
    if p.exists(): _d(Image(str(p), width=800))

In [ ]:
# contoh prediksi model terbaik pada gambar test
trained = df[df.tag.str.startswith(('s2', 's3', 's4'))]
best_tag = trained.loc[trained.mAP50.idxmax(), 'tag']
print('Model terbaik (hasil training):', best_tag)
best_model = YOLO(str(Path(PROJECT)/best_tag/'weights'/'best.pt'))
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, img in zip(axes, random.sample(test_imgs, 3)):
    ax.imshow(best_model.predict(str(img), conf=0.25, imgsz=IMGSZ, verbose=False)[0].plot()[..., ::-1]); ax.axis('off')
plt.tight_layout(); plt.show()

from google.colab import files
shutil.copy(Path(PROJECT)/best_tag/'weights'/'best.pt', '/content/best_tomatod.pt')
files.download('/content/best_tomatod.pt')
files.download('/content/hasil_perbandingan.csv')

## Cara membaca hasil

- **mAP50** dan **mAP50-95** adalah metrik utama. Lihat juga AP per kelas: `semi-ripe` paling sedikit datanya (117 objek di test), jadi biasanya paling rendah.
- Skenario 1 hanya mengukur lokalisasi class-agnostic, bukan kematangan, jadi tidak bisa dibandingkan langsung dengan skenario 2-4. Cukup tunjukkan bahwa model COCO belum bisa membedakan kematangan.
- Yang dibandingkan: apakah head-only cukup, apakah membuka lebih banyak layer menambah mAP, dan apakah pretrained mengalahkan scratch pada data kecil (189 gambar train).
- Test set hanya 55 gambar dan berasal dari sesi yang sama dengan train, jadi selisih kecil antar skenario belum tentu bermakna.